## Movies Dataset 2024 - Grupo 9
<hr style="border:1px solid gray">

# 1. Exploración y Compresión de Datos

## 1.1. Carga del Dataset

Importar librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import kagglehub
from scipy import stats as st
from scipy.stats import describe
from pathlib import Path
import shutil

Importar dataset

In [ ]:
# Create data directory if it doesn't exist
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Download dataset (latest version by default)
download_path = kagglehub.dataset_download("asaniczka/tmdb-movies-dataset-2023-930k-movies")

# Copy to local data directory
csv_file = next(Path(download_path).glob("*.csv"))
local_path = data_dir / csv_file.name
shutil.copy2(csv_file, local_path)

print(f"Dataset copied to: {local_path}")

Carga del dataset

In [ ]:
# Cargamos el dataset desde el directorio local de datos
data_path = "data/TMDB_movie_dataset_v11.csv"
movies = pd.read_csv(data_path)

## 1.2. Descripción del dataset IMDB Movies 2024


| Variable              | Descripción                                                                                  |
|-----------------------|----------------------------------------------------------------------------------------------|
| `id`                  | Unique identifier for each movie. (type: int)                                                |
| `title`               | Title of the movie (usually in English or the display title). (type: str)                    |
| `vote_average`        | Average vote or rating given by viewers (typically 0–10 scale). (type: float)                |
| `vote_count`          | Total count of votes received for the movie. (type: int)                                     |
| `status`              | Current status of the movie (Released, Rumored, Post Production, etc.). (type: str)          |
| `release_date`        | Date when the movie was released (format usually YYYY-MM-DD). (type: str)                    |
| `revenue`             | Total revenue generated by the movie (in USD; 0 often means unknown/missing). (type: int)    |
| `runtime`             | Duration of the movie in minutes (0 usually means unknown). (type: int)                      |
| `adult`               | Indicates if the movie is only for adult audiences (true/false). (type: bool)                |
| `backdrop_path`       | Relative path to the backdrop image for the movie . (type: str)                              |
| `budget`              | Estimated production budget of the movie (USD; 0 often means unknown/missing). (type: int64) |
| `homepage`            | Official website URL of the movie (empty string or null if not available). (type: str)       |
| `imdb_id`             | IMDb identifier for the movie (format: tt followed by 7–8 digits). (type: str)               |
| `original_language`   | ISO 639-1 code of the original language of the movie (e.g., 'en', 'fr', 'es'). (type: str)   |
| `original_title`      | Original title of the movie (in its original language). (type: str)                          |
| `overview`            | Brief plot summary or synopsis of the movie. (type: str)                                     |
| `popularity`          | Popularity score assigned by TMDB (higher = more popular). (type: float64)                   |
| `poster_path`         | Relative path to the poster image for the movie (prepend https://image.tmdb.org) (type: str) |
| `tagline`             | Short marketing tagline or slogan of the movie (often empty). (type: str)                    |
| `genres`              | List of genres (usually as comma-separated string or JSON in some datasets). (type: str)     |
| `production_companies`| Name(s) of the main production company/companies (often comma-separated or JSON) (type: str) |
| `production_countries`| Country/countries where the movie was produced (often comma-separated or JSON). (type: str)  |
| `spoken_languages`    | Language(s) spoken in the movie (comma-separated or JSON. ISO codes and names). (type: str)  |
| `keywords`            | Descriptive keywords/tags linked with the movie (often comma-separated or JSON). (type: str) |



#### Tipos de Variables

Nota: 2 son columnas de ID: id, imdb_id

Numericas Continuas (4)
- vote_count
- revenue
- runtime
- budget

Numericas Discretas (2)
- vote_average
- popularity

Categoricas Nominales Binarias (1)
- adult

Categoricas Nominales (15)
- title
- status
- release_date
- backdrop_path
- homepage
- budget
- runtime
- original_language
- original_title
- overview
- poster_path
- tangline
- genres
- production_companies
- production_countries
- spoken_languages

### Dimensiones del dataset original

In [ ]:
print(f"Dimensiones del dataset original: {movies.shape}")
# exclude columns w/o valuable data (e.g. backdrop_path, imdb_id,..)
columns = ['title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'budget', 'original_language', 'popularity', 'genres', 'production_companies', 'production_countries', 'keywords']
movies[columns].head()

Informacion de columnas y tipo de datos

In [ ]:
movies.info()

In [ ]:
movies[columns].describe()

## 1.3 Identificar patrones generales y distribuciones

### Análisis de las variables numéricas

_Nota: hay columnas con muchos valores en 0_

IMDB usa el 0 para representar "unknown", no el valor 0. Por eso usamos alternativamente las siguientes medidas para obtener resultados útiles:
- `field_valid = movies['field'][movies['field'] > 0]` # subset de valores mayores a 0
- `movies['field'].replace(0, np.nan).mode()` # reemplazar 0 por NaN

In [ ]:
movies['budget'].median() # example w/o filtering 0s

In [ ]:
budget_valid = movies['budget'][movies['budget'] > 0]
budget_valid.median() # iltering 0s

In [ ]:
movies['popularity'].mean() # example w/o filtering 0s

In [ ]:
movies['popularity'].replace(0, np.nan).mean() # example filtering 0s

#### Definir sets válidos para mostrar medidas no afectada por valores faltantes

In [ ]:
budget_valid = movies['budget'][movies['budget'] > 0]
revenue_valid = movies['revenue'][movies['revenue'] > 0]
popularity_valid = movies['popularity'][movies['popularity'] > 0]
vote_count_valid = movies['vote_count'][movies['vote_count'] > 0]
vote_average_valid = movies['vote_average'][movies['vote_average'] > 0]
runtime_valid = movies['runtime'][movies['runtime'] > 0]

#### Max/Mins

In [ ]:
# Maximum budget
max_budget = budget_valid.max()
print(f"Maximum budget: ${max_budget:,}")

# Minimum budget
min_budget = budget_valid.min()
print(f"Minimum budget: ${min_budget:,}")

# Maximum revenue
max_revenue = revenue_valid.max()
print(f"Maximum revenue: ${max_revenue:,}")

# Minimum revenue
min_revenue = revenue_valid.min()
print(f"Minimum revenue: ${min_revenue:,}")

# Maximum popularity
max_popularity = popularity_valid.replace(0, np.nan).max()
print(f"Maximum popularity: {max_popularity:,}")

# Minimum popularity
min_popularity = popularity_valid.replace(0, np.nan).min()
print(f"Minimum popularity: {min_popularity:,}")

# Maximum vote_average
max_vote_average = vote_average_valid.replace(0, np.nan).max()
print(f"Maximum vote_average: {max_vote_average:,}")

# Minimum vote_average
min_vote_average = vote_average_valid.replace(0, np.nan).min()
print(f"Minimum vote_average: {min_vote_average:,}")

# Maximum vote_count
max_vote_count = vote_count_valid.replace(0, np.nan).max()
print(f"Maximum vote_count: {max_vote_count:,}")

# Minimum vote_count
min_vote_count = vote_count_valid.replace(0, np.nan).min()
print(f"Minimum vote_count: {min_vote_count:,}")

# Maximum runtime
max_runtime = runtime_valid.replace(0, np.nan).max()
print(f"Maximum runtime: {max_runtime:,}")

# Minimum vote_count
min_runtime = runtime_valid.replace(0, np.nan).min()
print(f"Minimum runtime: {min_runtime:,}")

#### Medidas de tendencia central: media, mediana y moda


#### Media

In [ ]:
np.mean(budget_valid.replace(0, np.nan))    # Budget

In [ ]:
np.mean(revenue_valid.replace(0, np.nan)) # Revenue

In [ ]:
np.mean(popularity_valid) # Popularity

In [ ]:
np.mean(vote_count_valid.replace(0, np.nan)) # Vote count

In [ ]:
np.mean(vote_average_valid.replace(0, np.nan)) # Vote average

In [ ]:
np.mean(runtime_valid.replace(0, np.nan)) # Runtime

#### Mediana

In [ ]:
np.median(budget_valid.replace(0, np.nan))  # Budget

In [ ]:
np.median(revenue_valid) # Revenue

In [ ]:
np.median(popularity_valid) # Popularity

In [ ]:
np.median(vote_count_valid) # Vote count

In [ ]:
np.median(vote_average_valid) # Vote average

In [ ]:
np.median(runtime_valid) # Runtime

#### Moda

In [ ]:
budget_valid.mode()[0]     # Budget

In [ ]:
revenue_valid.mode()[0]     # Revenue

In [ ]:
popularity_valid.mode()[0]     # Popularity

In [ ]:
vote_count_valid.mode()[0]     # Vote count

In [ ]:
vote_average_valid.mode()[0]     # Vote average

In [ ]:
runtime_valid.mode()[0]     # Runtime

### Análisis de variables categóricas

Analizamos algunas variables categóricas del dataset como idioma original, géneros y países de producción para entender su distribución dentro del dataset.



In [ ]:
# Lista de columnas categóricas relevantes del dataset
categorical_columns = [
    "original_language",
    "genres",
    "production_countries",
    "production_companies",
    "keywords"
]

# Mostrar ejemplos de estas columnas
movies[categorical_columns].head()

#### Idioma original de las películas

En primer lugar analizamos la variable `original_language`, que indica el idioma original de cada película.

Este análisis permite observar qué idiomas están más representados en el dataset y si existe una concentración importante en algunos idiomas particulares.

Para ello calculamos la cantidad de películas por idioma

In [ ]:
# Contar cuántas películas hay por idioma
language_counts = movies["original_language"].value_counts()

# Cantidad total de idiomas distintos
print("Cantidad de idiomas distintos:", movies["original_language"].nunique())

# Mostrar los 10 idiomas más frecuentes
language_counts.head(10)

### Géneros de las películas

La variable `genres` contiene los géneros asociados a cada película. En el dataset esta columna está almacenada como un string separado por comas (por ejemplo: `Action, Science Fiction, Adventure`).

Para poder analizar los géneros de forma individual, primero convertimos este string en una lista y luego utilizamos `explode()` para crear una fila por cada género asociado a una película.

De esta forma podemos calcular la frecuencia de cada género y analizar su popularidad promedio dentro del dataset.

In [ ]:
movies["genres"].iloc[0]

In [ ]:
# La columna genres está almacenada como un string separado por comas
# Ejemplo: "Action, Science Fiction, Adventure"

# Primero convertimos ese string en una lista separando por coma

movies["genres"] = movies["genres"].fillna("").apply(lambda x: [g.strip() for g in x.split(",") if g])

# Ahora cada fila tiene una lista de géneros

movies["genres"].head()

In [ ]:
# "Aplanar" la lista de géneros
genres = movies.explode("genres")

# Verificar resultado
genres[["title","genres"]].head()

In [ ]:
# Contar géneros
genre_counts = genres["genres"].value_counts()

genre_counts.head(10)

In [ ]:
# Calcular la popularidad promedio para cada género
genre_popularity = genres.groupby("genres")["popularity"].mean().sort_values(ascending=False)

genre_popularity.head(10)

### Países de producción

También analizamos los países de producción para identificar qué países participan con mayor frecuencia en la producción de películas del dataset.

In [ ]:
# La columna production_countries también está almacenada como un string separado por comas
# Convertimos el string en una lista

movies["production_countries"] = movies["production_countries"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c]
)

movies["production_countries"].head()

In [ ]:
# Aplanar la lista de países
countries = movies.explode("production_countries")

countries[["title","production_countries"]].head()

In [ ]:
# Conteo de países
country_counts = countries["production_countries"].value_counts()

country_counts.head(10)

### Productoras

Analizamos las productoras involucradas en las películas para identificar cuáles participan con mayor frecuencia en la producción cinematográfica.

In [ ]:
movies["production_companies"].head()

In [ ]:
# Aplanar la lista de productoras
companies = movies.explode("production_companies")

companies[["title", "production_companies"]].head()

In [ ]:
#Conteo de productoras
company_counts = companies["production_companies"].value_counts()

company_counts.head(10)

### Palabras clave (Keywords)

Las palabras clave permiten identificar los temas o conceptos más frecuentes en las películas del dataset.

In [ ]:
movies["keywords"].head()

In [ ]:
# Convertimos keywords a lista (igual que genres y production_countries)
movies['keywords'] = movies['keywords'].fillna('').apply(
    lambda x: [k.strip() for k in x.split(',') if k.strip()]
)

# Aplanar la lista de keywords
keywords_exploded = movies.explode('keywords')

keywords_exploded[['title', 'keywords']].head()

In [ ]:
# Conteo de keywords
keyword_counts = keywords_exploded['keywords'].value_counts()

keyword_counts.head(10)

#### Resumen:

#####   Cantidad de filas: 1.407.834
#####   Cantidad de columnas: 24

### Hallazgos

El análisis de variables categóricas muestra algunos patrones interesantes en el dataset:

- **Idioma original:** El inglés aparece como el idioma predominante en el dataset, lo cual es consistente con la fuerte presencia de producciones de la industria cinematográfica estadounidense. Otros idiomas aparecen con una frecuencia considerablemente menor.

- **Géneros:** Los géneros más frecuentes en el dataset son *Drama*, *Documentary* y *Comedy*. Esto indica que una gran parte del conjunto de películas analizadas pertenece a categorías narrativas o documentales, mientras que otros géneros aparecen con menor frecuencia.

- **Popularidad por género:** Al analizar la popularidad promedio por género se observa que *Adventure* y *Action* presentan los valores más altos de popularidad promedio. Esto sugiere que, aunque no sean los géneros más numerosos en el dataset, este tipo de películas tiende a generar mayor interés o visibilidad.

- **Países de producción:** La producción cinematográfica del dataset está fuertemente concentrada en algunos países. *United States of America* aparece como el principal país productor, seguido por *Japan*, *France* y *United Kingdom*. Esto refleja el peso de estas industrias cinematográficas dentro del conjunto de datos analizado.

- **Productoras:** Algunas compañías aparecen con mayor frecuencia dentro del dataset, lo que indica que ciertas productoras participan en múltiples proyectos y tienen un rol importante dentro del conjunto de películas analizado. Entre las productoras con mayor cantidad de películas se encuentran *BBC*, *Warner Bros. Pictures*, *Columbia Pictures* y *Universal Pictures*.

- **Palabras clave:** El análisis de keywords muestra que existen ciertos temas o características recurrentes en las películas del dataset. Entre las más frecuentes aparecen términos relacionados con formatos de producción (por ejemplo *short film*), características de dirección (*woman director*) o el origen de la obra (*based on novel or book*). Esto sugiere que el dataset incluye una amplia variedad de producciones y temáticas

En conjunto, el análisis de estas variables permite comprender mejor la estructura del dataset y las características principales de las películas incluidas. A su vez, el proceso de conversión de strings a listas y el uso de `explode()` permitió analizar correctamente estas variables categóricas.

## 1.4 Identificar errores, outliers (anomalías), valores faltantes y su tipo (MCAR, MAR, MNAR).

### Películas duplicadas

In [ ]:
# Dataset information
print(f"Amount of movies (rows): {len(movies)}")
print(f"Amount of unique movies: {movies['id'].nunique()}")
print(f"Amount of columns: {len(movies.columns)}")

# Check for duplicate IDs
duplicate_ids = movies['id'].duplicated().sum()
if duplicate_ids > 0:
    print(f"\n Found {duplicate_ids} duplicate movie IDs!")
else:
    print("\nAll movie IDs are unique")

### Clasificación de Valores Faltantes

Identificamos las columnas con valores nulos y los clasificamos según su naturaleza:

- **MCAR (Missing Completely at Random):** Los valores faltantes no dependen de ninguna otra variable. Ejemplo: `runtime` (pueden ser omisiones aleatorias en la carga).
- **MAR (Missing at Random):** La probabilidad de que falte un dato depende de otras variables observadas. Ejemplo: `tagline` o `keywords` (películas menos populares o producciones independientes suelen carecer de estos metadatos con más frecuencia).
- **MNAR (Missing Not at Random):** Los valores faltantes dependen del valor del propio dato faltante. Ejemplo: `revenue` o `budget` (películas con menor éxito financiero podrían omitir reportar estos datos).

In [ ]:
# Resumen de valores nulos
missing_values = movies.isnull().sum()
missing_percent = (missing_values / len(movies)) * 100
missing_df = pd.DataFrame({'Nulos': missing_values, 'Porcentaje': missing_percent})
print("Columnas con valores nulos:")
print(missing_df[missing_df['Nulos'] > 0].sort_values(by='Porcentaje', ascending=False))

### Detección de Outliers

Un **outlier** (valor atípico) es una observación que se aleja significativamente del resto de los datos. Pueden deberse a errores de carga, casos excepcionales reales, o datos faltantes codificados como valores numéricos (como los ceros en `budget`).

Analizamos las variables numéricas que entran al modelo:

- `runtime` y `vote_average`: se visualizan directamente, ya que no tienen ceros como dato faltante.
- `budget` y `revenue`: se analizan **solo sobre valores conocidos (> 0)**. El valor `0` en estas columnas codifica "dato desconocido" en TMDB, no una película con costo o recaudación nula. Incluirlos distorsionaría el boxplot y los percentiles de capping.

#### Boxplots de runtime y vote_average

In [ ]:
# Boxplots de runtime y vote_average para detectar outliers visualmente.
cols_to_check = ['runtime', 'vote_average']

plt.figure(figsize=(12, 5))
for i, col in enumerate(cols_to_check, 1):
    plt.subplot(1, 2, i)
    sns.boxplot(y=movies[col])
    plt.title(f'Outliers en {col}')

plt.tight_layout()
plt.show()

#### Interpretación — runtime y vote_average (pre-tratamiento)

- **runtime**: la gran mayoría de las películas tiene una duración de entre ~60 y ~180 minutos (la caja es estrecha y baja). Sin embargo, hay muchos puntos por encima de los bigotes, incluyendo valores de más de 5.000 y hasta 14.400 minutos. Estos son claramente errores de carga.

- **vote_average**: el boxplot muestra que el dataset completo incluye muchas películas con rating 0, lo que aplana la distribución. Esto se debe a que aún no aplicamos el filtro de `vote_count > 100`. Hay también valores en 10 (rating perfecto) que podrían ser casos con muy pocos votos.

#### Boxplots de budget y revenue filtrando ceros
_Visualizar solo valores conocidos permite ver la distribución real sin distorsión._


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(y=budget_valid, ax=axes[0])
axes[0].set_title('Outliers en budget (sin ceros)')
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))

sns.boxplot(y=revenue_valid, ax=axes[1])
axes[1].set_title('Outliers en revenue (sin ceros)')
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))

plt.tight_layout()
plt.show()

#### Interpretación — budget y revenue (pre-tratamiento, sin ceros)

- **budget**: la caja está completamente aplastada cerca de $0, lo que indica que la mayoría de las películas con presupuesto conocido tienen valores bajos o moderados. Hay varios outliers por encima de $300M, con un máximo de $1.000M.

- **revenue**: patrón similar al budget. La caja está cerca de $0 con la mediana por debajo de $100M, y hay outliers que llegan hasta $5.000M (Avengers: Endgame, Avatar).

## 2. Técnicas de Visualización

### 2.1. Histogramas de Variables Numéricas

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))

# --- Budget ---
ax[0,0].hist(np.log1p(budget_valid), bins=10, color='cornflowerblue', edgecolor='black')
ax[0,0].set_title('Budget (log scale)')

# Nice original-scale tick labels
def log1p_inverse(y):
    return np.expm1(y)

ax[0,0].xaxis.set_major_formatter(ticker.FuncFormatter(
    lambda y, pos: f"${log1p_inverse(y)/1e6:.0f}M" if y >= 13 else f"${log1p_inverse(y)/1e3:.0f}K"
))

# --- Revenue ---
# valid_revenue = movies['revenue'][movies['revenue'] > 0]  # also drop negative
ax[0,1].hist(np.log1p(revenue_valid), bins=10, color='lightgreen', edgecolor='black')
ax[0,1].set_title('Revenue (log scale)')
ax[0,1].xaxis.set_major_formatter(ticker.FuncFormatter(
    lambda y, pos: f"${log1p_inverse(y)/1e6:.0f}M"
))

# --- Popularity ---
ax[0,2].hist(np.log1p(popularity_valid), bins=10, color='salmon', edgecolor='black')
ax[0,2].set_title('Popularity (log scale)')
ax[0,2].xaxis.set_major_formatter(ticker.FuncFormatter(
    lambda y, pos: f"{log1p_inverse(y):.0f}"
))

# --- Vote Average ---
ax[1,0].hist(vote_average_valid, bins=10, color='yellow', edgecolor='black')
ax[1,0].set_title('Vote Average')

# --- Vote Count ---
ax[1,1].hist(np.log1p(vote_count_valid), bins=10, color='lightblue', edgecolor='black')
ax[1,1].set_title('Vote Count (log scale)')

# --- Rubntime ---
ax[1,2].hist(np.log1p(runtime_valid), bins=10, color='pink', edgecolor='black')
ax[1,2].set_title('Runtime (log scale)')

plt.tight_layout()
plt.show()

### 2.2. Graficos de Variables Categoricas


#### Movies per moth

In [ ]:
#standarize dates
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')

movies['release_month'] = movies['release_date'].dt.month
ax = movies['release_month'].hist(bins=24)
ax.set_title('Movies per month')
plt.show()

#### Idioma original de las películas

Visualizamos los más frecuentes mediante un gráfico de barras

In [ ]:
# Gráfico de barras con los idiomas más frecuentes
language_counts.head(10).plot(kind="bar")

plt.title("Top 10 idiomas en el dataset")
plt.xlabel("Idioma")
plt.ylabel("Cantidad de películas")
plt.show()

#### Paises por Producción

In [ ]:
#Gráfico de países
country_counts.head(10).plot(kind="bar")

plt.title("Top 10 países productores de películas")
plt.xlabel("País")
plt.ylabel("Cantidad de películas")
plt.show()

#### Mayores productoras de Peliculas

In [ ]:
#Gráfico de productoras
company_counts.head(10).plot(kind="bar")

plt.title("Top 10 productoras")
plt.xlabel("Productora")
plt.ylabel("Cantidad de películas")
plt.show()

#### Keywords mas frecuentes

In [ ]:
#Gráfico de keywords
keyword_counts.head(10).plot(kind="bar")

plt.title("Keywords más frecuentes en el dataset")
plt.xlabel("Keyword")
plt.ylabel("Frecuencia")
plt.show()

#### Géneros de Películas

In [ ]:
# Visualizar los géneros más frecuentes
genre_counts.head(10).plot(kind="bar")

plt.title("Top 10 géneros de películas")
plt.xlabel("Género")
plt.ylabel("Cantidad de películas")
plt.show()

In [ ]:
# Visualizar qué géneros tienen mayor popularidad promedio
genre_popularity.head(10).plot(kind="bar")

plt.title("Popularidad promedio por género")
plt.xlabel("Género")
plt.ylabel("Popularidad promedio")
plt.show()

### Películas duplicadas

In [ ]:
# Visualizar información del dataset
fig, ax = plt.subplots(figsize=(10, 6))

# Datos
categorias = ['Filas totales', 'IDs únicos', 'Duplicados']
valores = [len(movies), movies['id'].nunique(), len(movies) - movies['id'].nunique()]
colores = ['#3498db', '#2ecc71', '#e74c3c']

# Crear gráfico de barras
bars = ax.bar(categorias, valores, color=colores)

# Configurar etiquetas y título
ax.set_title('Información del Dataset', fontsize=14, fontweight='bold')
ax.set_ylabel('Cantidad', fontsize=12)

# Formatear el eje Y para mostrar valores completos
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# Ajustar escala del eje Y
max_val = max(valores)
ax.set_ylim(0, max_val * 1.1)

# Agregar valores sobre las barras
for bar, valor in zip(bars, valores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + max_val * 0.01,
             f'{valor:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Agregar grid para mejor lectura
ax.grid(axis='y', alpha=0.3)

# Rotar etiquetas si es necesario
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

# 3. Problema de ML Supervisado

## 3.1. Descripcion del Problema de Aprendizaje Supervisado

Nuestro objetivo es preparar las variables predictoras para estimar el `vote_average` de una película. Es un problema de regresión. Filtramos las variables que no aportan información predictiva (identificadores, nombres, slogans) y nos enfocamos en metadatos como géneros, idiomas y métricas de producción.

Se plantea un escenario inspirado en plataformas de streaming como Netflix, donde se dispone de información de películas antes de su estreno. En este contexto, el objetivo es predecir el vote_average que tendrá una película a partir de sus características previas al lanzamiento, tales como género, presupuesto, duración, idioma o país de producción.

Este enfoque tiene aplicaciones reales en la industria, como la toma de decisiones sobre adquisición de contenido, priorización en catálogos o estimación del engagement esperado.

A partir de esta definición, se excluyen variables que no estarían disponibles en el momento previo al estreno o que podrían introducir sesgo de información. Por ejemplo, la variable popularity no se considera, ya que es calculada en función de la interacción de los usuarios una vez que la película ya fue lanzada.

De esta manera, se asegura que el modelo utilice únicamente información disponible en un escenario realista de predicción previa al estreno.

## 3.2. Variable Target

### vote_average

# 4. Procesamiento y limpieza

## 4.1 Realizar una limpieza general del dataset, eliminando o corrigiendo datos inconsistentes o irrelevantes.

En esta sección prepararemos el dataset filtrando aquellas películas con pocos votos para asegurar la confiabilidad de nuestra variable objetivo: `vote_average`.

### Limpieza de películas duplicadas

Se eliminan las filas con peliculas duplicadas

In [ ]:
# remove duplicates
movies = movies.drop_duplicates(subset=['id'], keep='first')

### 4.2 Tratamiento de outliers

La estrategia elegida es el **capping por percentiles (p1–p99)**: los valores por debajo del percentil 1 se elevan a ese límite, y los valores por encima del percentil 99 se recortan a ese límite. Esto **no elimina filas**, solo acota los valores extremos dentro de un rango razonable.

Se eligió esta técnica sobre otras (como eliminar filas o imputar con la mediana) porque:
- Preserva la cantidad total de datos
- Mantiene la variabilidad interna del dataset
- Es robusta para distribuciones muy sesgadas como `budget` y `revenue`

**Primer paso**: reemplazar los ceros de `budget` y `revenue` por `NaN`, ya que representan datos faltantes y no deben participar en el cálculo de percentiles.

In [ ]:
# Tratamiento de outliers — clipping p1-p99 sobre variables numéricas del modelo
def cap_outliers(df, col):
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)
    return df

# quantile() y clip() ignoran NaN por defecto, por lo que budget y revenue
# se calculan solo sobre valores conocidos. El replace(0, NaN) del paso anterior
# ya convirtió las columnas a float64, lo que permite almacenar NaN.
for col in ['runtime', 'vote_average', 'budget', 'revenue']:
    cap_outliers(movies_filtered, col)

In [ ]:
# Función de capping: recorta los valores de una columna al rango [p1, p99].
# quantile() ignora NaN por defecto, por lo que budget y revenue se calculan
# solo sobre valores reales (los NaN del paso anterior no afectan los percentiles).
def cap_outliers(df, col):
    lower = df[col].quantile(0.01)  # percentil 1: límite inferior
    upper = df[col].quantile(0.99)  # percentil 99: límite superior
    df[col] = df[col].clip(lower, upper)  # clip reemplaza valores fuera del rango
    return df

# runtime y vote_average: capping directo sobre todos los valores
for col in ['runtime', 'vote_average']:
    cap_outliers(movies_filtered, col)

# budget y revenue: capping solo sobre valores conocidos.
# Aunque los ceros ya fueron reemplazados por NaN (celda anterior), usamos notna()
# como salvaguarda explícita para dejar claro que los NaN se preservan intactos.
for col in ['budget', 'revenue']:
    mask = movies_filtered[col].notna()
    lower = movies_filtered.loc[mask, col].quantile(0.01)
    upper = movies_filtered.loc[mask, col].quantile(0.99)
    movies_filtered.loc[mask, col] = movies_filtered.loc[mask, col].clip(lower, upper)

In [ ]:
# Función de capping: recorta los valores de una columna al rango [p1, p99].
# quantile() ignora NaN por defecto, por lo que budget y revenue se calculan
# solo sobre valores reales (los NaN del paso anterior no afectan los percentiles).
def cap_outliers(df, col):
    lower = df[col].quantile(0.01)  # percentil 1: límite inferior
    upper = df[col].quantile(0.99)  # percentil 99: límite superior
    df[col] = df[col].clip(lower, upper)  # clip reemplaza valores fuera del rango
    return df

# runtime y vote_average: capping directo sobre todos los valores
for col in ['runtime', 'vote_average']:
    cap_outliers(movies_filtered, col)

# budget y revenue: capping solo sobre valores conocidos.
# Se convierten a float64 primero porque quantile() devuelve float y clip() no puede
# asignar floats a una columna int64 cuando hay NaN de por medio.
for col in ['budget', 'revenue']:
    movies_filtered[col] = movies_filtered[col].astype('float64')
    mask = movies_filtered[col].notna()
    lower = movies_filtered.loc[mask, col].quantile(0.01)
    upper = movies_filtered.loc[mask, col].quantile(0.99)
    movies_filtered.loc[mask, col] = movies_filtered.loc[mask, col].clip(lower, upper)

In [ ]:
# Tratamiento de outliers — clipping p1-p99 sobre variables numéricas del modelo
def cap_outliers(df, col):
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)
    return df

# runtime y vote_average: clipping directo (no tienen ceros como datos faltantes)
for col in ['runtime', 'vote_average']:
    cap_outliers(movies_filtered, col)

# budget y revenue: clipping solo sobre valores conocidos (cero = dato faltante, se preserva)
for col in ['budget', 'revenue']:
    mask = movies_filtered[col] > 0
    lower = movies_filtered.loc[mask, col].quantile(0.01)
    upper = movies_filtered.loc[mask, col].quantile(0.99)
    movies_filtered.loc[mask, col] = movies_filtered.loc[mask, col].clip(lower, upper)

#### Visualización luego del tratamiento de outliers

In [ ]:
# Visualización de outliers con boxplots (post-tratamiento)
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# runtime y vote_average: visualización directa
for ax, col in zip(axes[:2], ['runtime', 'vote_average']):
    sns.boxplot(y=movies_filtered[col], ax=ax)
    ax.set_title(f'{col} (post-tratamiento)')

# budget y revenue: solo valores conocidos para no distorsionar
for ax, col in zip(axes[2:], ['budget', 'revenue']):
    valid = movies_filtered[movies_filtered[col] > 0][col]
    sns.boxplot(y=valid, ax=ax)
    ax.set_title(f'{col} (sin ceros, post-tratamiento)')
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))

plt.tight_layout()
plt.show()

#### Interpretación — post-tratamiento de outliers

- **runtime**: la distribución quedó acotada entre ~25 y ~175 minutos. La caja concentra la mayoría de los valores alrededor de los 90-100 minutos, que es la duración típica de una película. Los valores extremos de miles de minutos fueron eliminados.

- **vote_average**: la distribución mejoró notablemente. Ahora el rango va de ~4.5 a ~8.0, lo que refleja películas con ratings confiables (resultado del filtro de `vote_count > 100` más el capping). La mediana queda alrededor de 6.5.

- **budget (sin ceros)**: los valores extremos de más de $800M fueron recortados al percentil 99 (~$260M). La caja sigue siendo compacta en valores bajos, con una cola larga hacia arriba, lo cual es esperable dado que pocas películas tienen presupuestos muy altos.

- **revenue (sin ceros)**: similar a budget. Los valores de más de $3.000M fueron recortados al percentil 99 (~$800M). La distribución sigue siendo sesgada hacia la derecha, pero sin los casos extremos que distorsionaban el análisis.

En todos los casos, el tratamiento redujo el impacto de los valores extremos sin eliminar observaciones del dataset.

### 4.3 Realizar el split del dataset (ej: train y test).

Para evitar el sesgo de información (data leakage), realizamos la división en conjuntos de entrenamiento y prueba antes de aplicar transformaciones que dependan de la distribución de los datos.

In [ ]:
from sklearn.model_selection import train_test_split

features_cols = ['genres', 'original_language', 'production_companies', 'production_countries',
                 'runtime', 'budget', 'revenue']
target = 'vote_average'

X = movies_filtered[features_cols].copy()
y = movies_filtered[target].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# 5. Feature Engineering

En esta fase creamos nuevas variables a partir de las existentes y preparamos las features para el modelado.

Se simplificaron algunas variables categóricas que presentaban una gran cantidad de valores posibles, como el idioma y el país de producción.
En particular, las variables `original_language` y `production_countries` presentaban una gran cantidad de categorías distintas.

Para abordar este problema, en lugar de trabajar con todas las categorías originales, se agruparon en clases más generales (por ejemplo, inglés vs. otros idiomas, o Estados Unidos vs. resto del mundo), con el objetivo de facilitar el modelado y evitar generar una gran cantidad de variables.
Esta simplificación permite mantener información relevante sobre el origen de las películas, al mismo tiempo que reduce la dimensionalidad del dataset y mejora la interpretabilidad de los resultados.

Importante: el feature engineering se aplica **por separado** sobre `X_train` y `X_test` para evitar data leakage.


In [ ]:
# Feature engineering sobre X_train y X_test por separado

def apply_feature_engineering(X):
    X = X.copy()

    # Convertir genres a lista si no lo es
    X['genres'] = X['genres'].apply(
        lambda x: x if isinstance(x, list) else [g.strip() for g in str(x).split(',') if g.strip()]
    )

    # Idioma: binario inglés vs otros
    X['is_english'] = X['original_language'].apply(lambda x: 1 if x == 'en' else 0)

    # País: binario USA vs resto
    X['is_usa'] = X['production_countries'].apply(
        lambda x: 1 if 'United States of America' in str(x) else 0
    )

    # Tomar solo el primer género
    X['genres'] = X['genres'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown'
    )

    # Tomar solo la primera productora
    X['production_companies'] = X['production_companies'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0
        else (x.split(',')[0].strip() if isinstance(x, str) and x else 'Unknown')
    )

    X.drop(columns=['original_language', 'production_countries'], inplace=True)
    return X

X_train = apply_feature_engineering(X_train)
X_test = apply_feature_engineering(X_test)

print('X_train shape:', X_train.shape)
print(X_train.head())


In [ ]:
X_train['is_english'].value_counts().plot(kind='bar')
plt.title("Películas en inglés vs otros idiomas (train set)")
plt.xticks([0,1], ['Otros', 'Inglés'], rotation=0)
plt.show()

In [ ]:
X_train['is_usa'].value_counts().plot(kind='bar')
plt.title("Películas de USA vs resto del mundo (train set)")
plt.xticks([0,1], ['Resto', 'USA'], rotation=0)
plt.show()

# 6. Codificación y Escalado

Aplicamos One-Hot Encoding a las variables categóricas y estandarizamos las numéricas mediante un Pipeline de Scikit-Learn.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Identificamos tipos de columnas
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'str']).columns.tolist()

print('Columnas numéricas:', num_cols)
print('Columnas categóricas:', cat_cols)

# Definimos transformadores
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# max_categories=20 agrupa productoras/géneros poco frecuentes en 'infrequent'
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=20))
])

# Combinamos en un Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Ajustamos sobre train, transformamos ambos
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Nombres de columnas para el DataFrame resultante
feature_names = (num_cols +
                 preprocessor.named_transformers_['cat']
                 .named_steps['onehot'].get_feature_names_out(cat_cols).tolist())

X_train_df = pd.DataFrame(X_train_proc, columns=feature_names)
print(f'Shape post-encoding: {X_train_df.shape}')
X_train_df.head()


# 7. Reducción de Dimensionalidad

Aplicamos técnicas de selección de features (basadas en correlación) y reducción de dimensionalidad (PCA) para simplificar nuestro conjunto de datos manteniendo la mayor parte de la información.

### 7.1 Análisis de Correlación

Identificamos variables con baja correlación con el objetivo o alta correlación entre sí (redundancia).

In [ ]:
# Correlación sobre variables numéricas (pre-encoding, post-FE)
# is_english e is_usa son int64 tras el feature engineering
num_cols_corr = [c for c in X_train.select_dtypes(include=['int64', 'float64']).columns]
corr_df = X_train[num_cols_corr].copy()
corr_df['vote_average'] = y_train

plt.figure(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlación de Variables Numéricas')
plt.show()


### 7.2 Aplicación de PCA (Análisis de Componentes Principales)

Reducimos el espacio de características procesado a un conjunto de componentes principales.

In [ ]:
from sklearn.decomposition import PCA

# Aplicamos PCA sin reducir inicialmente para ver la varianza explicada
pca = PCA()
X_pca = pca.fit_transform(X_train_df)

# Varianza acumulada
explained_variance_ratio = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', linestyle='--')
plt.axhline(y=0.9, color='r', linestyle='-')
plt.title('Varianza Explicada Acumulada por Componentes Principales')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Acumulada')
plt.grid()
plt.show()

# Seleccionamos componentes que expliquen el 90% de la varianza
n_components_90 = np.argmax(explained_variance_ratio >= 0.9) + 1
print(f"Número de componentes para explicar el 90% de la varianza: {n_components_90}")

### 7.3 Análisis de Loadings

Para interpretar los componentes principales, analizamos la contribución (pesos) de las variables originales a los dos primeros componentes.

In [ ]:
# Obtenemos los pesos (loadings) de los componentes
# pca.components_.T tiene shape (n_features, n_components)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=feature_names
)

# Top 10 variables que más contribuyen al PC1
plt.figure(figsize=(10, 5))
loadings['PC1'].abs().sort_values(ascending=False).head(10).plot(kind='bar', color='skyblue')
plt.title('Top 10 Contribuyentes al Componente Principal 1 (PC1)')
plt.ylabel('Carga (Valor Absoluto)')
plt.tight_layout()
plt.show()

# Top 10 variables que más contribuyen al PC2
plt.figure(figsize=(10, 5))
loadings['PC2'].abs().sort_values(ascending=False).head(10).plot(kind='bar', color='salmon')
plt.title('Top 10 Contribuyentes al Componente Principal 2 (PC2)')
plt.ylabel('Carga (Valor Absoluto)')
plt.tight_layout()
plt.show()


### 7.4 Visualización del Espacio de Características Reducido

Graficamos las películas en el nuevo espacio definido por PC1 y PC2, coloreando por `vote_average`.

In [ ]:
# Scatter plot de PC1 vs PC2 coloreado por la variable objetivo
plt.figure(figsize=(12, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train.values, cmap='viridis', alpha=0.5, s=10)
plt.colorbar(label='vote_average')
plt.title('Proyección de Películas en PC1 y PC2')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.show()

# 8. Conclusiones Finales

Tras completar el análisis y preparación de los datos de TMDB, podemos concluir:

1. **Estructura del Dataset:** El conjunto de datos presenta una alta dimensionalidad y una gran cantidad de variables categóricas. La limpieza y extracción del metadato principal permitió simplificar el análisis manteniendo la información clave.
2. **Calidad de Datos:** La clasificación de nulos permitió entender la naturaleza de los datos faltantes (MAR/MNAR). El filtrado por `vote_count > 100` fue fundamental para garantizar que la variable objetivo sea robusta.
3. **Representatividad mediante PCA:** El análisis demostró que es posible reducir drásticamente el número de variables (logrando explicar el 90% de la varianza) utilizando componentes principales, lo cual optimiza el dataset para futuras etapas de modelado.
4. **Visualización:** La proyección en el espacio de componentes principales permite analizar la distribución general de las películas. Sin embargo, no se observan agrupamientos claramente definidos según el vote_average, lo que sugiere que las variables consideradas no separan de forma evidente las películas en función de su puntuación.